In [0]:
# Databricks notebook source
# THIS AS VIBE CODED!!!!
# =============================================================================
# 05_create_genie_agent.py
#
# Creates or updates the U.S. Energy Inflation Intelligence Genie Agent.
#
# This notebook is intended to run INSIDE Databricks.
#
# No Databricks SDK is required.
#
# Authentication is taken directly from the current Databricks notebook
# context using the temporary notebook token.
# =============================================================================

import json
import uuid
import requests


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# Replace this with the ID of Serverless SQL Warehouse.
WAREHOUSE_ID = "<Replace me with your serverless sql warehouse id>"

METRIC_VIEW = (
    "workspace.us_electricity.energy_inflation_metrics"
)

GENIE_TITLE = (
    "U.S. Energy Inflation Intelligence"
)

GENIE_DESCRIPTION = (
    "Analyze historical U.S. electricity prices, electricity inflation, "
    "headline inflation, core inflation, and Henry Hub natural gas prices "
    "using governed monthly economic metrics sourced from FRED."
)


# =============================================================================
# 2. VALIDATE CONFIGURATION
# =============================================================================

if (
    not WAREHOUSE_ID
    or WAREHOUSE_ID.startswith("<")
):
    raise ValueError(
        "WAREHOUSE_ID is not configured. "
        "Replace <REPLACE_WITH_SQL_WAREHOUSE_ID> "
        "with a Pro or Serverless SQL Warehouse ID."
    )

if METRIC_VIEW.count(".") != 2:
    raise ValueError(
        "METRIC_VIEW must use a three-level Unity Catalog name. "
        f"Received: {METRIC_VIEW}"
    )


# =============================================================================
# 3. GET DATABRICKS WORKSPACE AUTHENTICATION
# =============================================================================
#
# Databricks provides the workspace URL and a temporary notebook token
# through the notebook context.
#
# Do not print API_TOKEN.
# =============================================================================

context = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
)

API_ROOT = context.apiUrl().get()
API_TOKEN = context.apiToken().get()

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}


# =============================================================================
# 4. DETERMINE DEFAULT GENIE WORKSPACE LOCATION
# =============================================================================
#
# Genie requires a parent workspace folder.
#
# Use the current Databricks user's workspace folder by default.
# =============================================================================

current_user = (
    spark.sql("SELECT current_user() AS user")
    .first()["user"]
)

PARENT_PATH = f"/Workspace/Users/{current_user}"

print(f"Genie Agent location: {PARENT_PATH}")


# =============================================================================
# 6. API HELPER
# =============================================================================

def genie_api_request(
    method,
    endpoint,
    *,
    params=None,
    payload=None
):
    """
    Make an authenticated request to the Databricks Genie API.
    """

    url = f"{API_ROOT}{endpoint}"

    response = requests.request(
        method=method,
        url=url,
        headers=HEADERS,
        params=params,
        json=payload,
        timeout=60
    )

    if not response.ok:
        raise RuntimeError(
            f"Databricks Genie API request failed.\n"
            f"Method: {method}\n"
            f"Endpoint: {endpoint}\n"
            f"Status: {response.status_code}\n"
            f"Response: {response.text}"
        )

    if response.text:
        return response.json()

    return None


# =============================================================================
# 7. DETERMINISTIC GENIE CONFIGURATION IDS
# =============================================================================
#
# Genie serialized_space version 2 requires IDs to be:
#
# - 32 characters
# - lowercase hexadecimal
#
# Using UUID5 makes IDs deterministic across notebook reruns.
# =============================================================================

ID_NAMESPACE = uuid.UUID(
    "fd9492d1-2276-4ef3-967e-67de49fc37bc"
)


def stable_id(category, name):
    """
    Generate a deterministic 32-character lowercase hex ID.
    """

    return uuid.uuid5(
        ID_NAMESPACE,
        f"{category}:{name}"
    ).hex


def sort_by_id(items):
    """
    Genie requires ID-based collections to already be sorted.
    """

    return sorted(
        items,
        key=lambda item: item["id"]
    )


# =============================================================================
# 8. SAMPLE QUESTIONS
# =============================================================================

SAMPLE_QUESTIONS = [
    (
        "What is the latest U.S. electricity "
        "inflation rate?"
    ),
    (
        "Is electricity inflation currently higher "
        "than headline inflation?"
    ),
    (
        "How have electricity inflation and headline "
        "inflation compared since 2020?"
    ),
    (
        "When was the electricity inflation premium highest?"
    ),
    (
        "How have natural gas prices and electricity "
        "inflation moved since 2020?"
    ),
    (
        "Has the relationship between electricity inflation "
        "and headline inflation changed over time?"
    )
]


sample_questions = sort_by_id([
    {
        "id": stable_id(
            "sample_question",
            question
        ),
        "question": [question]
    }
    for question in SAMPLE_QUESTIONS
])


# =============================================================================
# 9. GENIE BUSINESS INSTRUCTIONS
# =============================================================================

TEXT_INSTRUCTIONS = [

    (
        "This Genie Agent analyzes monthly U.S. energy and inflation "
        "data using governed measures from the configured metric view."
    ),

    (
        "Use measures from the metric view whenever possible instead "
        "of recalculating governed metrics."
    ),

    (
        "Electricity inflation means the year-over-year percentage "
        "change in the U.S. Electricity Consumer Price Index."
    ),

    (
        "Electricity price means the average U.S. electricity price "
        "per kilowatt-hour. Electricity price is a price level and is "
        "different from electricity inflation."
    ),

    (
        "Headline inflation, overall inflation, consumer inflation, "
        "and CPI inflation refer to year-over-year headline CPI."
    ),

    (
        "Core inflation means the year-over-year percentage change "
        "in CPI excluding food and energy."
    ),

    (
        "Electricity inflation premium means electricity inflation "
        "minus headline inflation and is measured in percentage points."
    ),

    (
        "Natural gas price refers to the monthly Henry Hub natural "
        "gas spot price."
    ),

    (
        "When a user asks for a latest or current value, use the most "
        "recent month containing a non-null value for that metric."
    ),

    (
        "Clearly distinguish percentages from percentage-point "
        "differences when comparing inflation rates."
    ),

    (
        "The dataset contains national U.S. monthly data. Do not imply "
        "that state-level or customer-level electricity data is available."
    ),

    (
        "Do not interpret correlation as proof of causation."
    ),

    (
        "Do not claim that natural gas prices caused electricity "
        "inflation based solely on these observational data. Describe "
        "relationships as associations, patterns, or correlations."
    ),

    (
        "Prefer monthly time-series analysis unless the user explicitly "
        "requests another time aggregation."
    )
]


text_instructions = sort_by_id([
    {
        "id": stable_id(
            "text_instruction",
            "energy_inflation_business_rules"
        ),
        "content": TEXT_INSTRUCTIONS
    }
])


# =============================================================================
# 10. VERIFIED EXAMPLE SQL
# =============================================================================

SQL_LATEST_ELECTRICITY_INFLATION = f"""
SELECT
    month,
    electricity_inflation_yoy
FROM (
    SELECT
        month,
        MEASURE(electricity_inflation_yoy)
            AS electricity_inflation_yoy
    FROM {METRIC_VIEW}
    GROUP BY month
)
WHERE electricity_inflation_yoy IS NOT NULL
ORDER BY month DESC
LIMIT 1
""".strip()


SQL_LATEST_ELECTRICITY_VS_HEADLINE = f"""
SELECT
    month,
    electricity_inflation_yoy,
    headline_inflation_yoy,
    electricity_inflation_premium
FROM (
    SELECT
        month,

        MEASURE(electricity_inflation_yoy)
            AS electricity_inflation_yoy,

        MEASURE(headline_inflation_yoy)
            AS headline_inflation_yoy,

        MEASURE(electricity_inflation_premium)
            AS electricity_inflation_premium

    FROM {METRIC_VIEW}

    GROUP BY month
)
WHERE electricity_inflation_yoy IS NOT NULL
  AND headline_inflation_yoy IS NOT NULL

ORDER BY month DESC
LIMIT 1
""".strip()


SQL_INFLATION_SINCE_2020 = f"""
SELECT
    month,

    MEASURE(electricity_inflation_yoy)
        AS electricity_inflation_yoy,

    MEASURE(headline_inflation_yoy)
        AS headline_inflation_yoy

FROM {METRIC_VIEW}

WHERE month >= DATE '2020-01-01'

GROUP BY month

ORDER BY month
""".strip()


SQL_HIGHEST_PREMIUM = f"""
SELECT
    month,

    MEASURE(electricity_inflation_premium)
        AS electricity_inflation_premium

FROM {METRIC_VIEW}

GROUP BY month

ORDER BY electricity_inflation_premium DESC

LIMIT 10
""".strip()


SQL_NATURAL_GAS_VS_ELECTRICITY = f"""
SELECT
    month,

    MEASURE(natural_gas_price_yoy)
        AS natural_gas_price_yoy,

    MEASURE(electricity_inflation_yoy)
        AS electricity_inflation_yoy

FROM {METRIC_VIEW}

WHERE month >= DATE '2020-01-01'

GROUP BY month

ORDER BY month
""".strip()


SQL_CORRELATION_TREND = f"""
SELECT
    month,

    MEASURE(electricity_headline_correlation_24m)
        AS electricity_headline_correlation_24m

FROM {METRIC_VIEW}

GROUP BY month

ORDER BY month
""".strip()


EXAMPLE_SQL_DEFINITIONS = [

    {
        "question":
            "What is the latest U.S. electricity inflation rate?",
        "sql":
            SQL_LATEST_ELECTRICITY_INFLATION
    },

    {
        "question":
            "Is electricity inflation currently higher "
            "than headline inflation?",
        "sql":
            SQL_LATEST_ELECTRICITY_VS_HEADLINE
    },

    {
        "question":
            "Show electricity inflation versus headline "
            "inflation since 2020.",
        "sql":
            SQL_INFLATION_SINCE_2020
    },

    {
        "question":
            "When was the electricity inflation premium highest?",
        "sql":
            SQL_HIGHEST_PREMIUM
    },

    {
        "question":
            "How have natural gas prices and electricity "
            "inflation moved since 2020?",
        "sql":
            SQL_NATURAL_GAS_VS_ELECTRICITY
    },

    {
        "question":
            "How has the 24-month relationship between electricity "
            "inflation and headline inflation changed over time?",
        "sql":
            SQL_CORRELATION_TREND
    }
]


example_question_sqls = sort_by_id([
    {
        "id": stable_id(
            "example_sql",
            definition["question"]
        ),
        "question": [
            definition["question"]
        ],
        "sql": [
            definition["sql"]
        ]
    }
    for definition in EXAMPLE_SQL_DEFINITIONS
])


# =============================================================================
# 11. BENCHMARK QUESTIONS
# =============================================================================

BENCHMARK_DEFINITIONS = [

    {
        "question":
            "What is the latest electricity inflation rate?",
        "sql":
            SQL_LATEST_ELECTRICITY_INFLATION
    },

    {
        "question":
            "How does the latest electricity inflation rate "
            "compare with headline inflation?",
        "sql":
            SQL_LATEST_ELECTRICITY_VS_HEADLINE
    },

    {
        "question":
            "Show electricity inflation and headline inflation "
            "by month since 2020.",
        "sql":
            SQL_INFLATION_SINCE_2020
    },

    {
        "question":
            "Which ten months had the highest electricity "
            "inflation premium?",
        "sql":
            SQL_HIGHEST_PREMIUM
    },

    {
        "question":
            "Show natural gas year-over-year price change and "
            "electricity inflation since 2020.",
        "sql":
            SQL_NATURAL_GAS_VS_ELECTRICITY
    },

    {
        "question":
            "Show the 24-month rolling correlation between "
            "electricity inflation and headline inflation.",
        "sql":
            SQL_CORRELATION_TREND
    }
]


benchmark_questions = sort_by_id([
    {
        "id": stable_id(
            "benchmark",
            definition["question"]
        ),

        "question": [
            definition["question"]
        ],

        "answer": [
            {
                "format": "SQL",
                "content": [
                    definition["sql"]
                ]
            }
        ]
    }

    for definition in BENCHMARK_DEFINITIONS
])


# =============================================================================
# 12. BUILD GENIE SERIALIZED CONFIGURATION
# =============================================================================
#
# Databricks requires serialized_space version 2 for new agents.
# =============================================================================

serialized_config = {

    "version": 2,

    "config": {
        "sample_questions": sample_questions
    },

    "data_sources": {

        "metric_views": [
            {
                "identifier": METRIC_VIEW,

                "description": [
                    (
                        "Governed semantic model for monthly U.S. "
                        "electricity prices, electricity inflation, "
                        "headline inflation, core inflation, Henry Hub "
                        "natural gas prices, inflation spreads, and "
                        "rolling correlations."
                    )
                ]
            }
        ]
    },

    "instructions": {

        "text_instructions":
            text_instructions,

        "example_question_sqls":
            example_question_sqls
    },

    "benchmarks": {
        "questions":
            benchmark_questions
    }
}


# Genie requires metric view identifiers to already be sorted.
serialized_config["data_sources"]["metric_views"] = sorted(
    serialized_config["data_sources"]["metric_views"],
    key=lambda item: item["identifier"]
)


# IMPORTANT:
#
# serialized_space itself must be a JSON STRING.
serialized_space = json.dumps(
    serialized_config,
    separators=(",", ":"),
    ensure_ascii=False
)


# =============================================================================
# 13. LIST EXISTING GENIE AGENTS
# =============================================================================

def find_spaces_by_exact_title(title):
    """
    Find existing Genie Agents matching an exact title.
    Handles pagination.
    """

    matches = []
    page_token = None

    while True:

        params = {
            "page_size": 100
        }

        if page_token:
            params["page_token"] = page_token

        result = genie_api_request(
            "GET",
            "/api/2.0/genie/spaces",
            params=params
        )

        for space in result.get("spaces", []):
            if space.get("title") == title:
                matches.append(space)

        page_token = result.get(
            "next_page_token"
        )

        if not page_token:
            break

    return matches


matching_spaces = find_spaces_by_exact_title(
    GENIE_TITLE
)


if len(matching_spaces) > 1:

    duplicate_ids = [
        space["space_id"]
        for space in matching_spaces
    ]

    raise RuntimeError(
        f"Found multiple Genie Agents named '{GENIE_TITLE}'. "
        f"Refusing to choose one automatically. "
        f"Matching IDs: {duplicate_ids}"
    )


# =============================================================================
# 14. CREATE OR UPDATE GENIE AGENT
# =============================================================================

if len(matching_spaces) == 0:

    print(
        f"No existing Genie Agent named "
        f"'{GENIE_TITLE}' found."
    )

    print(
        "Creating Genie Agent..."
    )

    create_payload = {

        "warehouse_id":
            WAREHOUSE_ID,

        "parent_path":
            PARENT_PATH,

        "serialized_space":
            serialized_space,

        "title":
            GENIE_TITLE,

        "description":
            GENIE_DESCRIPTION
    }


    deployed = genie_api_request(
        "POST",
        "/api/2.0/genie/spaces",
        payload=create_payload
    )

    action = "CREATED"


else:

    existing = matching_spaces[0]

    space_id = existing["space_id"]

    print(
        f"Existing Genie Agent found: {space_id}"
    )

    print(
        "Updating Genie Agent..."
    )


    # Retrieve the latest ETag before updating.
    existing_detail = genie_api_request(
        "GET",
        f"/api/2.0/genie/spaces/{space_id}",
        params={
            "include_serialized_space": "true"
        }
    )


    update_payload = {

        "warehouse_id":
            WAREHOUSE_ID,

        "parent_path":
            PARENT_PATH,

        "serialized_space":
            serialized_space,

        "title":
            GENIE_TITLE,

        "description":
            GENIE_DESCRIPTION,

        "etag":
            existing_detail.get("etag")
    }


    deployed = genie_api_request(
        "PATCH",
        f"/api/2.0/genie/spaces/{space_id}",
        payload=update_payload
    )

    action = "UPDATED"


# =============================================================================
# 15. VERIFY DEPLOYMENT
# =============================================================================

space_id = deployed["space_id"]

verified = genie_api_request(
    "GET",
    f"/api/2.0/genie/spaces/{space_id}",
    params={
        "include_serialized_space": "true"
    }
)


returned_serialized_space = verified.get(
    "serialized_space"
)

if not returned_serialized_space:
    raise RuntimeError(
        "Genie Agent was deployed, but serialized_space "
        "was not returned during verification."
    )


returned_config = json.loads(
    returned_serialized_space
)


if returned_config.get("version") != 2:
    raise RuntimeError(
        "Genie Agent deployment verification failed: "
        "serialized_space version is not 2."
    )



# =============================================================================
# 16. SUCCESS
# =============================================================================

print()
print("=" * 70)

print(
    f"GENIE AGENT {action} SUCCESSFULLY"
)

print("=" * 70)

print(
    f"Title:        {verified['title']}"
)

print(
    f"Space ID:     {verified['space_id']}"
)

print(
    f"Warehouse ID: {verified['warehouse_id']}"
)

print(
    f"Metric View:  {METRIC_VIEW}"
)

print(
    f"Parent Path:  {verified.get('parent_path')}"
)

print()
print(
    f"Sample Questions:      "
    f"{len(sample_questions)}"
)

print(
    f"Verified SQL Examples: "
    f"{len(example_question_sqls)}"
)

print(
    f"Benchmarks:            "
    f"{len(benchmark_questions)}"
)

print("=" * 70)